# 02_clean_market_data.ipynb
**DyPriZa – Limpieza y normalización del dataset de mercado (versión robusta)**

Este notebook **detecta automáticamente** el CSV de entrada más probable (dataset crudo) y genera como salida:
- `data/DyPriZa_MVP.csv`  ➜ dataset limpio para el feature engineering.

Si ya sabes el nombre exacto de tu CSV crudo, rellena `RAW_CSV_OVERRIDE` en la celda de parámetros.

In [ ]:

RAW_CSV_OVERRIDE =None 
DELIM =None 
OUTPUT_PATH ='data/DyPriZa_MVP.csv'

from pathlib import Path 
Path ('data').mkdir (exist_ok =True )


## 1) Localización del dataset de entrada (auto-detección)

In [ ]:
import os ,time 

def find_csv_candidates ():
    candidates =[]
    for root ,dirs ,files in os .walk ('.',topdown =True ):
        for name in files :
            if name .lower ().endswith ('.csv'):
                path =os .path .join (root ,name )
                lower =name .lower ()

                skip =any (k in lower for k in [
                'mvp','feature','features','pred','prediction','predictions',
                'pricing','clean_market_data','_features','_pred'
                ])
                if skip :
                    continue 
                try :
                    stat =os .stat (path )
                    size =stat .st_size 
                    mtime =stat .st_mtime 
                except Exception :
                    size ,mtime =0 ,0 
                candidates .append ((path ,size ,mtime ))
    candidates .sort (key =lambda x :(x [2 ],x [1 ]),reverse =True )
    return candidates 

def choose_input_path (override =None ):
    if override and os .path .exists (override ):
        print (f"✅ Usando RAW_CSV_OVERRIDE: {override}")
        return override 
    cands =find_csv_candidates ()
    if not cands :
        raise FileNotFoundError ("No se encontraron CSVs crudos. Coloca tu dataset en ./data o en la raíz.")
    print ("🔎 Candidatos detectados (top 10):")
    for i ,(p ,s ,m )in enumerate (cands [:10 ]):
        print (f"[{i}] {p} | tamaño={s/1024:.1f} KB | mod={time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(m))}")
    chosen =cands [0 ][0 ]
    print (f"\n➡️ Seleccionado automáticamente: {chosen}")
    return chosen 

INPUT_PATH =choose_input_path (RAW_CSV_OVERRIDE )


## 2) Carga del CSV con autodetección de delimitador

In [ ]:
import pandas as pd 

def try_load_csv (path ,delim_hint =None ,nrows_test =2000 ):

    if delim_hint :
        return pd .read_csv (path ,delimiter =delim_hint )

    df_semicolon =pd .read_csv (path ,delimiter =';')
    df_comma =pd .read_csv (path ,delimiter =',')

    def score (df ):
        cols =df .shape [1 ]
        null_rate =df .isnull ().mean ().mean ()
        return (cols ,-null_rate )
    return df_semicolon if score (df_semicolon )>score (df_comma )else df_comma 

data =try_load_csv (INPUT_PATH ,DELIM )
print ('✅ Cargado:',data .shape )
data .head (3 )


## 3) Normalización y tipado seguro

In [ ]:
data .columns =[c .strip ().lower ()for c in data .columns ]


c_fecha =next ((c for c in data .columns if c in ('fecha','date','ds')),None )
c_precio =next ((c for c in data .columns if c in ('precio','price')),None )
c_ocup =next ((c for c in data .columns if 'ocupa'in c or c =='occupancy'),None )
c_ciudad =next ((c for c in data .columns if c in ('ciudad','city','market')),None )
c_prop =next ((c for c in data .columns if c in ('propiedad','listing_id','property_id','id_propiedad')),None )

if c_fecha :
    data [c_fecha ]=pd .to_datetime (data [c_fecha ],errors ='coerce')
print ('🕒 Columna de fecha:',c_fecha )

if c_precio :data [c_precio ]=pd .to_numeric (data [c_precio ],errors ='coerce')
if c_ocup :data [c_ocup ]=pd .to_numeric (data [c_ocup ],errors ='coerce')

data .info ()


## 4) Limpieza: duplicados, nulos y outliers (precio)

In [ ]:

before =data .shape [0 ]
data =data .drop_duplicates ()
print ('🧹 Duplicados eliminados:',before -data .shape [0 ])


if c_precio :
    med_price =data [c_precio ].median (skipna =True )
    data [c_precio ]=data [c_precio ].fillna (med_price )
if c_ocup :
    med_occ =data [c_ocup ].median (skipna =True )
    data [c_ocup ]=data [c_ocup ].fillna (med_occ )


if c_precio :
    p1 ,p99 =data [c_precio ].quantile ([0.01 ,0.99 ])
    data [c_precio ]=data [c_precio ].clip (lower =p1 ,upper =p99 )
    print (f'✂️ Winsorization precio entre p1={p1:.2f} y p99={p99:.2f}')
else :
    print ('⚠️ No se encontró columna de precio; se omite winsorization')


## 5) Orden lógico y guardado del dataset limpio

In [ ]:
sort_cols =[c for c in [c_prop ,c_ciudad ,c_fecha ]if c ]
if sort_cols :
    data =data .sort_values (by =sort_cols )
    print ('↕️ Ordenado por:',sort_cols )

data .to_csv (OUTPUT_PATH ,index =False )
print ('✅ Guardado limpio en:',OUTPUT_PATH )
data .head (3 )
